# 11 — V1 External Manufacturer Maintenance Knowledge

**Research question.** Does *real* manufacturer maintenance-schedule knowledge
(`ridebase-ml/external_knowledge/`), added to the leakage-free v1.3 snapshot features,
meaningfully improve **next-service KM** prediction? Secondary: does it help **DAYS**?

This is **not** a dataset-generation experiment. RideBase Synthetic Dataset **v1.3 is FROZEN** —
source tables, targets, splits and noise are untouched. The external-knowledge CSVs are read
**read-only**. This notebook produces only *derived* ML features and runs a leakage-safe A/B test.

Contracts: NEXT-ANY-SERVICE raw days / raw km-delta · **observed-only** (censored snapshots are
never pseudo-labelled) · authoritative TRAIN/VALIDATION/TEST · TEST opened **once**, after the
config is frozen on VALIDATION · same default `HistGradientBoostingRegressor(random_state=42)`
for BASE and KNOWLEDGE so the delta is attributable to the knowledge features alone.

In [ ]:
"""11_v1_external_maintenance_knowledge — leakage-safe A/B test of real manufacturer
maintenance-schedule features on the FROZEN RideBase v1.3 next-service KM/DAYS regression.
Dataset generation experiment DEĞİLDİR: v1.3 source/target/split/noise değişmez.
External knowledge kaynak dosyaları read-only kullanılır; sadece derived ML feature üretilir."""
from pathlib import Path
from collections import OrderedDict
import json, os, platform, re, warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None
try:
    from autogluon.tabular import TabularPredictor
except Exception:
    TabularPredictor = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 160); pd.set_option("display.max_rows", 240)

SEED = 42
np.random.seed(SEED)
FAST = os.environ.get("RB_FAST") == "1"
N_PERM = 2 if FAST else 8
PERM_SAMPLE = 1500 if FAST else 6000
DATASET_VERSION = "1.3.0"

def find_project_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "notebooks").is_dir() and (c / "models").is_dir() and (c / "external_knowledge").is_dir():
            return c
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")

ROOT = find_project_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_3"
DERIVED = DATASET_ROOT / "derived_outputs"
KB = ROOT / "external_knowledge"
MODELS, OUTPUTS, REPORTS = ROOT / "models", ROOT / "outputs", ROOT / "reports"
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures" / "v1_external_knowledge"
for d in (MODELS, OUTPUTS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

DAYS_TOL = [15, 30, 60]
KM_TOL = [500, 1000, 1500, 2000]

def regression_metrics(y, pred, target):
    y = np.asarray(y, float); pred = np.asarray(pred, float); ae = np.abs(pred - y)
    out = {"mae": mean_absolute_error(y, pred), "median_ae": median_absolute_error(y, pred),
           "rmse": mean_squared_error(y, pred) ** 0.5, "r2": r2_score(y, pred),
           "bias": float(np.mean(pred - y)), "p90_ae": float(np.quantile(ae, .90)), "n": int(len(y))}
    for x in (DAYS_TOL if target == "DAYS" else KM_TOL):
        out[f"within_{x}"] = float(np.mean(ae <= x))
    return out

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES / name, dpi=140, bbox_inches="tight"); plt.close()

plt.style.use("seaborn-v0_8-whitegrid")
print("SETUP OK | FAST=", FAST, "| xgb", XGBRegressor is not None, "lgbm", LGBMRegressor is not None,
      "catboost", CatBoostRegressor is not None, "autogluon", TabularPredictor is not None)

## 1 · Dataset guard (v1.3 frozen) + observed-only contract
Assert `dataset_version == generator_version == 1.3.0`, `regression_calibration_experiment`,
the authoritative split `{TRAIN 32203, VALIDATION 4845, TEST 4470}` and the observed regression
mask `{TRAIN 25442, VALIDATION 1429, TEST 1282}`. Censored → excluded, never 0/-1/median/pseudo.

In [ ]:
# ---- v1.3 DATASET GUARD (frozen) + observed-only regression contract ----
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)
info = metadata["dataset"]
if info.get("dataset_version") != DATASET_VERSION or info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError("Yalnız RideBase Synthetic Dataset v1.3.0 kullanılabilir")
if metadata.get("regression_calibration_experiment") is not True:
    raise RuntimeError("regression_calibration_experiment flag beklenen değil")

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

EXPECTED_SPLIT = {"TRAIN": 32203, "VALIDATION": 4845, "TEST": 4470}
if split_manifest.set_index("snapshot_id").primary_time_split.value_counts().to_dict() != EXPECTED_SPLIT:
    raise RuntimeError("Split guard failed")
if len(snapshots) != 41518 or snapshots.snapshot_id.duplicated().any():
    raise RuntimeError("Snapshot guard failed")

tsel = targets[["snapshot_id", "target_event_observed", "is_right_censored", "days_to_next_service",
                "km_to_next_service", "target_km_valid", "next_service_type_code", "next_service_is_breakdown"]]
df = (snapshots.merge(tsel, on="snapshot_id", how="left", validate="one_to_one")
      .merge(split_manifest[["snapshot_id", "primary_time_split",
                             "next_service_regression_eligible_primary", "primary_label_cutoff_at"]],
             on="snapshot_id", how="left", validate="one_to_one"))
df["split"] = df["primary_time_split"].astype(str)
df["snapshot_at"] = pd.to_datetime(df["snapshot_at"])
df = df.set_index("snapshot_id", drop=False)

elig = df.next_service_regression_eligible_primary.astype(bool)
obs_mask = (elig & df.days_to_next_service.notna() & np.isfinite(df.days_to_next_service)
            & (df.days_to_next_service >= 0))       # observed-only: censored asla pseudo-label değil
km_mask = (obs_mask & df.km_to_next_service.notna() & np.isfinite(df.km_to_next_service)
           & (df.km_to_next_service >= 0) & (df.target_km_valid == 1))
EXPECTED_OBS = {"TRAIN": 25442, "VALIDATION": 1429, "TEST": 1282}
obs_counts = df.loc[obs_mask].groupby("split").size().to_dict()
if obs_counts != EXPECTED_OBS:
    raise RuntimeError(f"Observed regression contract mismatch: {obs_counts}")
print("DATASET_GUARD=PASS |", info["dataset_version"], "| split", EXPECTED_SPLIT, "| observed", obs_counts)
print("KM observed:", df.loc[km_mask].groupby("split").size().to_dict())

## 2 · BASE_v1_3 feature contract
The 146-feature leakage-safe set used by `10_v1_3_final_regression` (BASE + POLICY + RECENT_USAGE
+ HISTORICAL_INTERVAL + MAINTENANCE_HISTORY), same `ColumnTransformer` encoder, same default HGB.
This is **SET A**.

In [ ]:
# ---- BASE_v1_3 leakage-safe feature set (v1.3 final-regression contract, HGB default) ----
ID_COLS = ["snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id", "model_id", "model_name"]
META_COLS = ["snapshot_at", "snapshot_date", "feature_version", "data_origin", "generator_version",
             "random_seed", "scenario_id"]
SYNTHETIC_ONLY = list(metadata["feature_availability"]["SYNTHETIC_ONLY_EXCLUDED_FROM_FINAL_ML"])
PROD_DERIVABLE = set(metadata["feature_availability"]["PRODUCTION_DERIVABLE"])
TARGET_COLS = list(tsel.columns) + ["split", "primary_time_split",
                                    "next_service_regression_eligible_primary", "primary_label_cutoff_at"]

GRP = OrderedDict()
GRP["POLICY"] = ["policy_group", "policy_ready", "policy_interval_days", "policy_interval_km",
                 "current_policy_due_task_count", "total_policy_due_task_count",
                 "maintenance_overdue_days_pre_service", "maintenance_overdue_km_pre_service"]
GRP["RECENT_USAGE"] = ["recent_30d_km", "recent_60d_km", "recent_90d_km", "recent_180d_km",
                       "recent_km_per_day", "long_term_km_per_day", "recent_vs_long_term_usage_ratio"]
GRP["HISTORICAL_INTERVAL"] = ["previous_service_count", "previous_interval_days", "previous_interval_km",
                              "historical_interval_days_median", "historical_interval_days_std",
                              "historical_interval_km_median", "historical_interval_km_std",
                              "avg_service_interval_days", "avg_service_interval_km",
                              "rolling3_interval_days", "rolling3_interval_km",
                              "days_since_previous_service", "km_since_previous_service",
                              "avg_km_per_day_since_previous_service", "service_sequence"]
GRP["MAINTENANCE_HISTORY"] = [
    "historical_policy_delay_median_days", "historical_policy_delay_mean_days", "historical_on_time_rate",
    "previous_policy_delay_days", "periodic_service_count", "repair_service_count", "breakdown_service_count",
    "warranty_service_count", "appointment_service_count", "walkin_service_count", "services_last_90d",
    "services_last_365d", "cumulative_service_spend", "avg_service_spend", "total_task_count",
    "total_completed_task_count", "total_declined_task_count", "total_fault_task_count",
    "total_inspection_finding_task_count", "total_replace_task_count",
    "days_since_engine_task", "km_since_engine_task", "days_since_brakes_task", "km_since_brakes_task",
    "days_since_final_drive_task", "km_since_final_drive_task", "days_since_tires_wheels_task",
    "km_since_tires_wheels_task", "days_since_electrical_task", "km_since_electrical_task",
    "days_since_transmission_task", "km_since_transmission_task", "days_since_cooling_task",
    "km_since_cooling_task", "days_since_intake_task", "km_since_intake_task",
    "previous_failure_count", "service_odometer_regression_count_to_date"]
NON_FEATURE = set(ID_COLS + META_COLS + SYNTHETIC_ONLY + TARGET_COLS)
grouped = {c for v in GRP.values() for c in v}
GRP["BASE"] = [c for c in snapshots.columns if c not in NON_FEATURE and c not in grouped]
GRP.move_to_end("BASE", last=False)
BASE_FEATURES = [c for g in GRP.values() for c in g]
BASE_CAT = [c for c in BASE_FEATURES if df[c].dtype == "object"]
for c in [c for c in BASE_FEATURES if df[c].dtype == "bool"]:
    df[c] = df[c].astype("float64")
BASE_NUM = [c for c in BASE_FEATURES if c not in BASE_CAT]
print(f"BASE_v1_3 features: {len(BASE_FEATURES)} ({len(BASE_NUM)} num / {len(BASE_CAT)} cat)")

is_tr = (df.split == "TRAIN").to_numpy(); is_va = (df.split == "VALIDATION").to_numpy(); is_te = (df.split == "TEST").to_numpy()

def make_encoder(num, cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                          ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=25,
                                              sparse_output=False))]), cat),
    ], remainder="drop", verbose_feature_names_out=True)

y = {t: {s: df.loc[{"TRAIN": is_tr, "VALIDATION": is_va, "TEST": is_te}[s] & obs_mask.to_numpy(),
                    "days_to_next_service" if t == "DAYS" else "km_to_next_service"].to_numpy(float)
         for s in ("TRAIN", "VALIDATION", "TEST")} for t in ("DAYS", "KM")}
obs_of = {s: obs_mask.to_numpy()[{"TRAIN": is_tr, "VALIDATION": is_va, "TEST": is_te}[s]] for s in ("TRAIN", "VALIDATION", "TEST")}
split_mask = {"TRAIN": is_tr, "VALIDATION": is_va, "TEST": is_te}

## 3 · Knowledge-base guard (read-only)
Verify the KB artifacts exist and the quality report is PASS. Exact counts are read from
`quality_report.csv` (policies / task rows / source docs). Source-conflict count is reported.

In [ ]:
# ---- KNOWLEDGE BASE GUARD (read-only) ----
kb_policies = pd.read_csv(KB / "manufacturer_maintenance_policies.csv")
kb_tasks = pd.read_csv(KB / "manufacturer_maintenance_tasks.csv", dtype=str)
kb_mapping = pd.read_csv(KB / "model_mapping.csv")
kb_sources = pd.read_csv(KB / "sources.csv")
kb_coverage = pd.read_csv(KB / "model_coverage.csv")
kb_conflicts = pd.read_csv(KB / "source_conflicts.csv")
kb_quality = pd.read_csv(KB / "quality_report.csv").set_index("check")["value"].to_dict()

for name, dfk, mn in [("policies", kb_policies, 20), ("tasks", kb_tasks, 380),
                      ("sources", kb_sources, 18), ("model_mapping", kb_mapping, 20),
                      ("model_coverage", kb_coverage, 39)]:
    if len(dfk) < mn:
        raise RuntimeError(f"KB guard: {name} beklenenden az satır ({len(dfk)} < {mn})")

N_POLICY = int(kb_quality["total_policy_rows"]); N_TASK = int(kb_quality["total_task_rows"])
N_SOURCE = int(kb_quality["total_sources"])
assert len(kb_policies) == N_POLICY and len(kb_tasks) == N_TASK and len(kb_sources) == N_SOURCE
QUALITY_FAIL = {k: kb_quality[k] for k in
                ("duplicate_task_rows", "negative_km", "zero_repeat_interval", "rows_missing_source_id",
                 "orphan_source_id", "rows_missing_confidence", "rows_missing_evidence") if int(kb_quality[k]) != 0}
if QUALITY_FAIL:
    raise RuntimeError(f"KB quality report NOT PASS: {QUALITY_FAIL}")
src_ids = set(kb_sources.source_id)
if not set(kb_tasks.source_id).issubset(src_ids):
    raise RuntimeError("KB task source_id orphan")

N_CONFLICT = len(kb_conflicts)
CONF_DIST = {"HIGH": int(kb_quality["HIGH_confidence_rows"]), "MEDIUM": int(kb_quality["MEDIUM_confidence_rows"]),
             "LOW": int(kb_quality["LOW_confidence_rows"])}
COV = kb_coverage.set_index("model_id")
N_RESOLVED = int((COV.status == "RESOLVED").sum())
N_PARTIAL = int((COV.status == "PARTIAL").sum())
N_UNRESOLVED = int((COV.status == "UNRESOLVED").sum())
print(f"KB_GUARD=PASS | policies {N_POLICY} tasks {N_TASK} sources {N_SOURCE} | "
      f"conf {CONF_DIST} | conflicts {N_CONFLICT} | resolved {N_RESOLVED} partial {N_PARTIAL} unresolved {N_UNRESOLVED}")
print("QUALITY REPORT: PASS (0 duplicate / negative / zero-repeat / missing-source / orphan / missing-confidence / missing-evidence)")

## 4 · Model bridge — RideBase `model_id` → KB
Deterministic bridge for the models carrying an official schedule, plus the `model_mapping.csv`
match-quality metadata (`match_type`, `match_confidence`, `year_match`, `market_match`).
UNRESOLVED mappings are never used. `GENERATION_MISMATCH` / `VARIANT_MATCH` are kept **with a
flag**. Unresolved source-conflict `(model, task)` pairs are recorded so their task feature can
be nulled rather than forced to one km value.

In [ ]:
# ---- RideBase model_id  ->  KB (brand, model) + mapping quality ----
# Explicit, deterministic bridge for the 23 models that carry an official schedule.
MODEL_ID_TO_KB = {
    "BAJAJ_NS200": ("Bajaj", "Pulsar NS200"), "BAJAJ_NS125": ("Bajaj", "Pulsar NS125"),
    "BAJAJ_DOMINAR250": ("Bajaj", "Dominar 250"), "ROYALENFIELD_HUNTER350": ("Royal Enfield", "Hunter 350"),
    "YAMAHA_XSR125": ("Yamaha", "XSR125"), "YAMAHA_NMAX125": ("Yamaha", "NMAX 125"),
    "YAMAHA_XMAX250": ("Yamaha", "XMAX 250"), "KTM_390_DUKE": ("KTM", "390 Duke"),
    "CFMOTO_250NK": ("CFMOTO", "250NK"), "CFMOTO_650NK": ("CFMOTO", "650NK"),
    "TVS_RAIDER125": ("TVS", "Raider 125"), "TVS_APACHE_RTR200_4V": ("TVS", "Apache RTR 200 4V"),
    "TVS_JUPITER125": ("TVS", "Jupiter 125"), "SYM_JETX125": ("SYM", "Jet X 125"),
    "MONDIAL_50_UAG": ("Mondial", "50 UAG"), "MONDIAL_SFC_MINI_50EC": ("Mondial", "SFC MINI 50ec"),
    "MONDIAL_WING50I": ("Mondial", "Wing 50i"), "MONDIAL_EXON50": ("Mondial", "Exon 50"),
    "MONDIAL_TURISMO50I": ("Mondial", "Turismo 50i"), "KYMCO_XTOWN_CT250": ("KYMCO", "X-Town CT 250"),
    "HONDA_PCX125": ("Honda", "PCX125"), "HONDA_CB125F": ("Honda", "CB125F"), "HONDA_SH125I": ("Honda", "SH125i"),
}
# every bridged id must actually have policy + task rows
for mid, (b, m) in MODEL_ID_TO_KB.items():
    if not ((kb_policies.brand == b) & (kb_policies.model == m)).any():
        raise RuntimeError(f"bridge: no policy for {mid} -> ({b},{m})")

def _norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())
MAP_META = {}
_map_by_norm = {(_norm(r.ridebase_brand), _norm(r.ridebase_model)): r for r in kb_mapping.itertuples()}
CONF_SCORE = {"HIGH": 1.0, "MEDIUM": 0.6, "LOW": 0.3}
for mid, (b, m) in MODEL_ID_TO_KB.items():
    cov = COV.loc[mid]
    r = _map_by_norm.get((_norm(cov["brand"]), _norm(cov["model"])))
    if r is None:  # fall back to source-model name
        r = next((x for x in kb_mapping.itertuples() if _norm(x.source_model) == _norm(m)), None)
    if r is None:
        raise RuntimeError(f"bridge: no model_mapping row for {mid}")
    MAP_META[mid] = dict(
        match_type=r.match_type, match_confidence=r.match_confidence,
        year_match=r.year_match, market_match=r.market_match,
        mapping_confidence_score=CONF_SCORE.get(r.match_confidence, 0.3),
        market_match_exact=int(r.market_match == "EXACT"),
        year_match_exact=int(r.year_match in ("EXACT", "APPROXIMATE")),      # PARTIAL / mismatch -> 0
        variant_match_exact=int(r.match_type in ("EXACT", "NORMALIZED_EXACT")),
        status=cov["status"])
# conflict lookup: (kb_brand, kb_model, canonical_task_code) that is NOT resolved to one value
CONFLICT_KEYS = set()
CONFLICT_COUNT = {}
for r in kb_conflicts.itertuples():
    CONFLICT_COUNT[(r.brand, r.model)] = CONFLICT_COUNT.get((r.brand, r.model), 0) + 1
    if str(r.resolution_status).upper() not in ("PREFER_BS6_OFFICIAL", "RESOLVED"):
        CONFLICT_KEYS.add((r.brand, r.model, r.canonical_task_code))
print("model bridge:", len(MODEL_ID_TO_KB), "models | mapping meta ready | unresolved conflict keys:", CONFLICT_KEYS)
MAPPED_IDS = set(MODEL_ID_TO_KB)

## 5 · Leakage-safe feature engineering (vectorised per model)
Every feature below is derived **only** from `snapshot_odometer_km` + `motorcycle_age_years` +
the static manufacturer schedule. The actual next-service target is never read.

* **Policy-level** — `manufacturer_service_interval_km/months`, `first_service_km`, `service_rule_type`,
  `km_to_next_manufacturer_service` (model-specific grid `[fs, s2, s2+I, …]`, not a blind modulo),
  `km_since_previous_grid`, `service_stage` / `grid_index`, `overdue_km` (10 % grace),
  `due_now_flag`, `months/days_to_next`, `due_pressure = min(normalised km-remaining, normalised time-remaining)`.
* **Task-level** — `km_to_next_<TASK>` for 13 canonical tasks, **action-aware** (air-filter
  replace vs clean; spark-plug replace vs inspect); `tasks_due_next_{250,500,1000,2000,5000}km`;
  safety / replace / inspect due counts; `km_to_next_major_maintenance` (explicit rule = next due
  REPLACE among valve-clearance / coolant / brake-fluid / final-drive).
* **Time-based** — `months_to_next_{BRAKE_FLUID, COOLANT, SERVICE}`.
* **Coverage / match quality** — `manufacturer_knowledge_available`, `policy_available`,
  `task_knowledge_available`, task-coverage / HIGH-MED-LOW counts, `has_conflict`,
  `market/year/variant_match_exact`, `mapping_confidence_score`.

**NULL semantics.** A model with no knowledge → `*_available = 0` and every `km_to_next_* = NaN`
(**never 0**, since 0 km means "due now"). Four confidence/mapping frames are built:
HIGH+MEDIUM (primary), HIGH-only, HIGH+MEDIUM+LOW, and exact-mapping-only.

In [ ]:
# ---- leakage-safe manufacturer feature engineering — vectorised per model ----
# Every feature is derived ONLY from snapshot_odometer_km + motorcycle_age_years + the static
# manufacturer schedule. The actual next-service target is NEVER read here.
def _num(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else np.nan
    except (TypeError, ValueError):
        return np.nan

CANON_KEEP = ["ENGINE_OIL", "ENGINE_OIL_FILTER", "AIR_FILTER", "SPARK_PLUG", "BRAKE_FLUID", "DRIVE_CHAIN",
              "COOLANT", "VALVE_CLEARANCE", "TYRES", "FINAL_DRIVE", "FRONT_BRAKE", "BRAKE_PADS", "FUEL_SYSTEM"]
REPLACE_ACT = {"REPLACE", "DRAIN_REPLACE"}
INSPECT_ACT = {"INSPECT", "CHECK"}
MAJOR_CODES = {"VALVE_CLEARANCE", "COOLANT", "BRAKE_FLUID", "FINAL_DRIVE"}
DUE_WINDOWS = [250, 500, 1000, 2000, 5000]
# (canonical codes, allowed actions) -> emitted km_to_next_<name> feature
TASK_FEATURES = [
    ("ENGINE_OIL", {"ENGINE_OIL"}, {"REPLACE"}),
    ("ENGINE_OIL_FILTER", {"ENGINE_OIL_FILTER"}, {"REPLACE"}),
    ("AIR_FILTER_replace", {"AIR_FILTER"}, {"REPLACE"}),
    ("AIR_FILTER_clean", {"AIR_FILTER"}, {"CLEAN"}),
    ("SPARK_PLUG_replace", {"SPARK_PLUG"}, {"REPLACE"}),
    ("SPARK_PLUG_inspect", {"SPARK_PLUG"}, {"CLEAN", "INSPECT"}),
    ("BRAKE_FLUID", {"BRAKE_FLUID"}, {"DRAIN_REPLACE"}),
    ("DRIVE_CHAIN", {"DRIVE_CHAIN"}, {"LUBRICATE", "ADJUST"}),
    ("COOLANT", {"COOLANT"}, {"DRAIN_REPLACE"}),
    ("VALVE_CLEARANCE", {"VALVE_CLEARANCE"}, {"ADJUST", "INSPECT"}),
    ("FINAL_DRIVE", {"FINAL_DRIVE"}, {"REPLACE"}),
    ("FRONT_BRAKE", {"FRONT_BRAKE", "BRAKE_PADS"}, {"INSPECT"}),
    ("TYRES", {"TYRES"}, {"INSPECT"}),
    ("FUEL_SYSTEM", {"FUEL_SYSTEM"}, {"INSPECT", "REPLACE"}),
]
TIME_FEATURES = [("BRAKE_FLUID", {"BRAKE_FLUID"}, {"DRAIN_REPLACE"}),
                 ("COOLANT", {"COOLANT"}, {"DRAIN_REPLACE"})]


def _kmnext_mat(odo, fd, re_):
    """odo:(n,) fd,re_:(T,) -> (n,T) distance to next occurrence; NaN where re_ invalid & odo>=fd."""
    odo = odo[:, None]; fd = fd[None, :]; re_ = re_[None, :]
    fd_eff = np.where(np.isfinite(fd), fd, re_)
    before = fd_eff - odo
    base = np.where(np.isfinite(fd), fd, 0.0)
    with np.errstate(invalid="ignore"):
        rem = np.mod(odo - base, re_)
        after = np.where(rem > 0, re_ - rem, 0.0)
    out = np.where(odo < fd_eff, before, after)
    out = np.where(np.isfinite(fd_eff) | (np.isfinite(re_) & (re_ > 0)), out, np.nan)
    out = np.where((~(np.isfinite(re_) & (re_ > 0))) & (odo >= np.where(np.isfinite(fd_eff), fd_eff, np.inf)),
                   np.nan, out)          # one-time task already passed -> NaN (not recurring)
    return out


def _grid_vec(odo, fs, s2, interval):
    """odo:(n,). grid = [fs, s2, s2+I, s2+2I, ...]. -> (km_to_next, km_since_prev, grid_index) each (n,)."""
    fs, s2, interval = _num(fs), _num(s2), _num(interval)
    n = len(odo)
    if not np.isfinite(interval) or interval <= 0:
        return np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    anchor = s2 if np.isfinite(s2) else (fs if np.isfinite(fs) else 0.0)
    top = float(odo.max()) + interval * 2
    ks = np.arange(1, int(max(1, (top - anchor) / interval)) + 2)
    pts = np.array(sorted(set([p for p in (fs, s2) if np.isfinite(p)] + list(anchor + ks * interval))))
    j = np.searchsorted(pts, odo, side="left")
    j = np.clip(j, 0, len(pts) - 1)
    nxt = pts[j].copy()
    past = odo > nxt
    nxt = np.where(past, np.where(j + 1 < len(pts), pts[np.clip(j + 1, 0, len(pts) - 1)], nxt + interval), nxt)
    prev_j = np.clip(np.searchsorted(pts, odo, side="right") - 1, 0, len(pts) - 1)
    prev = np.where(odo >= pts[0], pts[prev_j], 0.0)
    gidx = np.array([float(np.sum(pts <= o)) for o in odo])
    return nxt - odo, odo - prev, gidx


def build_knowledge(conf_levels, mapping_types, tag):
    """Per-snapshot manufacturer feature frame for a confidence / mapping filter.
    conf_levels: subset of {'HIGH','MEDIUM','LOW'}. mapping_types: allowed match_type set or 'ALL'."""
    tk = kb_tasks[kb_tasks.confidence.isin(conf_levels)].copy()
    for c in ("first_due_km", "repeat_every_km", "repeat_every_months"):
        tk[c + "_n"] = tk[c].map(_num)
    tk["is_safety"] = tk.is_safety_critical.astype(str) == "1"
    tk["is_per"] = tk.is_periodic.astype(str) == "1"

    allowed = {mid for mid in MAPPED_IDS
               if mapping_types == "ALL" or MAP_META[mid]["match_type"] in mapping_types}
    age_months = (df["motorcycle_age_years"].astype(float) * 12.0).to_numpy()
    odo_all = df["snapshot_odometer_km"].astype(float).to_numpy()
    mid_all = df["model_id"].to_numpy()

    blocks = []
    for mid in sorted(allowed):
        pos = np.flatnonzero(mid_all == mid)
        if pos.size == 0:
            continue
        b, m = MODEL_ID_TO_KB[mid]
        pol = kb_policies[(kb_policies.brand == b) & (kb_policies.model == m)]
        rows = tk[(tk.brand == b) & (tk.model == m)]
        rows = rows[~rows.canonical_task_code.isin({c for (bb, mm, c) in CONFLICT_KEYS if bb == b and mm == m})]
        if pol.empty:
            continue
        pol = pol.iloc[0]
        meta = MAP_META[mid]
        odo = odo_all[pos]; agem = age_months[pos]
        blk = pd.DataFrame(index=df.index[pos])
        blk["mk_manufacturer_knowledge_available"] = 1
        blk["mk_manufacturer_policy_available"] = 1
        blk["mk_manufacturer_task_knowledge_available"] = int(len(rows) > 0)
        # ---- policy-level ----
        I_km = _num(pol.regular_service_interval_km); I_mo = _num(pol.regular_service_interval_months)
        fs = _num(pol.first_service_km_max) if np.isfinite(_num(pol.first_service_km_max)) else _num(pol.first_service_km_min)
        s2 = _num(pol.second_service_km_max) if np.isfinite(_num(pol.second_service_km_max)) else _num(pol.second_service_km_min)
        k_next, k_since, gidx = _grid_vec(odo, fs, s2, I_km)
        blk["mk_manufacturer_service_interval_km"] = I_km
        blk["mk_manufacturer_service_interval_months"] = I_mo
        blk["mk_manufacturer_first_service_km"] = fs
        blk["mk_manufacturer_service_rule_type"] = str(pol.service_rule_type)
        blk["mk_km_to_next_manufacturer_service"] = k_next
        blk["mk_km_since_previous_manufacturer_service_grid"] = k_since
        blk["mk_manufacturer_service_grid_index"] = gidx
        blk["mk_manufacturer_service_stage"] = np.clip(gidx, 0, 20)
        grace = 0.10 * I_km if np.isfinite(I_km) else np.nan
        blk["mk_manufacturer_overdue_km"] = np.maximum(0.0, k_since - grace) if np.isfinite(grace) else np.nan
        blk["mk_manufacturer_due_now_flag"] = ((k_next <= grace).astype(int) if np.isfinite(grace)
                                               else np.zeros(len(odo), int))
        fmo = _num(pol.first_service_months)
        if np.isfinite(I_mo) and I_mo > 0:
            anchor_mo = fmo if np.isfinite(fmo) else I_mo
            since_mo = np.where(agem >= anchor_mo, np.mod(agem - anchor_mo, I_mo), np.nan)
            m2n = np.where(np.isfinite(since_mo), I_mo - since_mo, anchor_mo - agem)
        else:
            m2n = np.full(len(odo), np.nan)
        blk["mk_months_to_next_manufacturer_service"] = m2n
        blk["mk_days_to_next_manufacturer_service"] = m2n * 30.44
        p_km = np.clip(k_next / I_km, 0, 1) if (np.isfinite(I_km) and I_km > 0) else np.full(len(odo), np.nan)
        p_mo = np.clip(m2n / I_mo, 0, 1) if (np.isfinite(I_mo) and I_mo > 0) else np.full(len(odo), np.nan)
        blk["mk_manufacturer_due_pressure"] = np.nanmin(np.vstack([p_km, p_mo]), axis=0)
        # ---- task-level ----
        fd = rows.first_due_km_n.to_numpy(float); rk = rows.repeat_every_km_n.to_numpy(float)
        code = rows.canonical_task_code.to_numpy(); act = rows.action.to_numpy()
        per = rows.is_per.to_numpy(); safe = rows.is_safety.to_numpy()
        if len(rows):
            M = _kmnext_mat(odo, fd, rk)               # (n, T)
        else:
            M = np.full((len(odo), 0), np.nan)
        for name, codes, acts in TASK_FEATURES:
            sel = np.array([(c in codes and a in acts) for c, a in zip(code, act)], bool)
            col = np.full(len(odo), np.nan)
            if sel.any():
                sub = M[:, sel]
                if np.isfinite(sub).any():
                    col = np.nanmin(np.where(np.isfinite(sub), sub, np.inf), axis=1)
                    col[~np.isfinite(col)] = np.nan
            blk[f"mk_km_to_next_{name}"] = col
        # due counts over periodic rows
        selp = per & np.array([np.isfinite(x) for x in fd + rk]) if len(rows) else np.zeros(0, bool)
        Mp = M[:, per] if len(rows) else np.full((len(odo), 0), np.nan)
        actp = act[per]; safep = safe[per]; codep = code[per]
        finite = np.isfinite(Mp)
        for w in DUE_WINDOWS:
            blk[f"mk_tasks_due_next_{w}km"] = np.sum(finite & (Mp >= 0) & (Mp <= w), axis=1)
        maxk = np.where(finite.any(axis=1), np.nanmax(np.where(finite, Mp, np.nan), axis=1), 1.0)
        blk["mk_tasks_overdue_count"] = np.sum(finite & (Mp <= 0.05 * np.maximum(1.0, maxk)[:, None]), axis=1)
        sm = np.array([a in REPLACE_ACT for a in actp], bool)
        im = np.array([a in INSPECT_ACT for a in actp], bool)
        mj = np.array([(c in MAJOR_CODES and a in REPLACE_ACT) for c, a in zip(codep, actp)], bool)
        blk["mk_safety_tasks_due_next_1000km"] = np.sum(finite[:, safep] & (Mp[:, safep] <= 1000), axis=1) if safep.any() else 0
        blk["mk_replace_tasks_due_next_1000km"] = np.sum(finite[:, sm] & (Mp[:, sm] <= 1000), axis=1) if sm.any() else 0
        blk["mk_inspect_tasks_due_next_1000km"] = np.sum(finite[:, im] & (Mp[:, im] <= 1000), axis=1) if im.any() else 0
        if mj.any():
            Mm = Mp[:, mj]; fm = finite[:, mj]
            km_major = np.where(fm.any(axis=1), np.nanmin(np.where(fm, Mm, np.nan), axis=1), np.nan)
            blk["mk_km_to_next_major_maintenance"] = km_major
            blk["mk_major_tasks_due_count"] = np.sum(fm & (Mm <= 2000), axis=1)
        else:
            blk["mk_km_to_next_major_maintenance"] = np.nan
            blk["mk_major_tasks_due_count"] = 0
        # time-based task features
        rmo = rows.repeat_every_months_n.to_numpy(float)
        for name, codes, acts in TIME_FEATURES:
            sel = np.array([(c in codes and a in acts and np.isfinite(x) and x > 0)
                            for c, a, x in zip(code, act, rmo)], bool)
            if sel.any():
                vv = np.stack([r_ - np.mod(agem, r_) for r_ in rmo[sel]], axis=1)
                blk[f"mk_months_to_next_{name}"] = np.min(vv, axis=1)
            else:
                blk[f"mk_months_to_next_{name}"] = np.nan
        blk["mk_months_to_next_SERVICE"] = m2n
        # coverage / match-quality
        blk["mk_manufacturer_task_coverage_count"] = rows.canonical_task_code.nunique()
        blk["mk_manufacturer_high_confidence_task_count"] = int((rows.confidence == "HIGH").sum())
        blk["mk_manufacturer_medium_confidence_task_count"] = int((rows.confidence == "MEDIUM").sum())
        blk["mk_manufacturer_low_confidence_task_count"] = int((rows.confidence == "LOW").sum())
        blk["mk_manufacturer_has_conflict"] = int(CONFLICT_COUNT.get((b, m), 0) > 0)
        blk["mk_manufacturer_task_conflict_count"] = CONFLICT_COUNT.get((b, m), 0)
        blk["mk_manufacturer_market_match_exact"] = meta["market_match_exact"]
        blk["mk_manufacturer_year_match_exact"] = meta["year_match_exact"]
        blk["mk_manufacturer_variant_match_exact"] = meta["variant_match_exact"]
        blk["mk_manufacturer_mapping_confidence_score"] = meta["mapping_confidence_score"]
        blocks.append(blk)

    K = pd.concat(blocks).reindex(df.index) if blocks else pd.DataFrame(index=df.index)
    flag_cols = [c for c in K.columns if ("count" in c or c.endswith("_flag") or c.endswith("_available")
                                          or c.endswith("_exact") or "tasks_due_next" in c
                                          or c.endswith("_1000km") or c == "mk_major_tasks_due_count"
                                          or c == "mk_tasks_overdue_count")]
    K[flag_cols] = K[flag_cols].fillna(0)
    if "mk_manufacturer_service_rule_type" in K:
        K["mk_manufacturer_service_rule_type"] = K["mk_manufacturer_service_rule_type"].fillna("NO_KNOWLEDGE")
    K["mk_manufacturer_mapping_confidence_score"] = K["mk_manufacturer_mapping_confidence_score"].fillna(0.0)
    K.attrs["tag"] = tag
    return K


KNOW = build_knowledge({"HIGH", "MEDIUM"}, "ALL", "HIGH_MEDIUM__ALL_MAPPINGS")
KNOW_HIGH = build_knowledge({"HIGH"}, "ALL", "HIGH_ONLY")
KNOW_ALLCONF = build_knowledge({"HIGH", "MEDIUM", "LOW"}, "ALL", "HIGH_MEDIUM_LOW")
KNOW_EXACT = build_knowledge({"HIGH", "MEDIUM"}, {"NORMALIZED_EXACT", "EXACT"}, "HIGH_MEDIUM__EXACT_ONLY")

KNOW_NUM = [c for c in KNOW.columns if KNOW[c].dtype != object]
KNOW_CAT = [c for c in KNOW.columns if KNOW[c].dtype == object]
avail = KNOW["mk_manufacturer_knowledge_available"].fillna(0).astype(int)
print(f"PRIMARY knowledge frame: {KNOW.shape[1]} features ({len(KNOW_NUM)} num / {len(KNOW_CAT)} cat)")
print("knowledge_available rows:", int(avail.sum()), "/", len(avail), f"({avail.mean()*100:.1f}%)")
_kmcols = [c for c in KNOW.columns if c.startswith("mk_km_to_next_") or c.startswith("mk_months_to_next_")]
_bad_null = int(((avail == 0) & KNOW[_kmcols].notna().any(axis=1)).sum())
if _bad_null:
    raise RuntimeError(f"NULL semantics violated: {_bad_null} non-covered rows have non-null due features")
print("NULL semantics: PASS (non-covered km_to_next_* are NaN, not 0)")

## 6 · Experiment feature sets + dedicated preprocessor
`SET_A` BASE · `SET_B` +policy · `SET_C` +policy+task · `SET_D` +policy+task+match/confidence ·
`SET_E` covered-models-only diagnostic · `SET_F` HIGH-only knowledge · `SET_G` HIGH+MEDIUM (=frozen primary).
A separate `models/v1_ext_maintenance_preprocessor.joblib` is fit on **TRAIN only**.

In [ ]:
# ---- experiment feature sets + separate train-only preprocessor ----
POLICY_KNOWLEDGE = [c for c in KNOW.columns if c in (
    "mk_manufacturer_service_interval_km", "mk_manufacturer_service_interval_months",
    "mk_manufacturer_first_service_km", "mk_manufacturer_service_rule_type",
    "mk_km_to_next_manufacturer_service", "mk_km_since_previous_manufacturer_service_grid",
    "mk_manufacturer_service_stage", "mk_manufacturer_service_grid_index", "mk_manufacturer_overdue_km",
    "mk_manufacturer_due_now_flag", "mk_months_to_next_manufacturer_service",
    "mk_days_to_next_manufacturer_service", "mk_manufacturer_due_pressure",
    "mk_manufacturer_policy_available")]
TASK_DUE_KNOWLEDGE = [c for c in KNOW.columns if (
    c.startswith("mk_km_to_next_") and c != "mk_km_to_next_manufacturer_service")
    or c.startswith("mk_months_to_next_BRAKE") or c.startswith("mk_months_to_next_COOLANT")
    or c.startswith("mk_months_to_next_SERVICE")
    or c.startswith("mk_tasks_due_next_") or c in (
    "mk_tasks_overdue_count", "mk_safety_tasks_due_next_1000km", "mk_replace_tasks_due_next_1000km",
    "mk_inspect_tasks_due_next_1000km", "mk_km_to_next_major_maintenance", "mk_major_tasks_due_count",
    "mk_manufacturer_task_knowledge_available")]
MATCH_QUALITY = [c for c in KNOW.columns if c in (
    "mk_manufacturer_knowledge_available", "mk_manufacturer_task_coverage_count",
    "mk_manufacturer_high_confidence_task_count", "mk_manufacturer_medium_confidence_task_count",
    "mk_manufacturer_low_confidence_task_count", "mk_manufacturer_has_conflict",
    "mk_manufacturer_task_conflict_count", "mk_manufacturer_market_match_exact",
    "mk_manufacturer_year_match_exact", "mk_manufacturer_variant_match_exact",
    "mk_manufacturer_mapping_confidence_score")]
KNOW_GROUPS = {"POLICY_KNOWLEDGE": POLICY_KNOWLEDGE, "TASK_DUE_KNOWLEDGE": TASK_DUE_KNOWLEDGE,
               "MATCH_QUALITY": MATCH_QUALITY}
assert not (set(POLICY_KNOWLEDGE) & set(TASK_DUE_KNOWLEDGE) & set(MATCH_QUALITY))

FRAMES = {"PRIMARY": KNOW, "HIGH_ONLY": KNOW_HIGH, "ALLCONF": KNOW_ALLCONF, "EXACT": KNOW_EXACT}
FEATURE_SETS = OrderedDict()
FEATURE_SETS["SET_A_BASE"] = dict(base=BASE_FEATURES, know=[], frame="PRIMARY", rows="ALL")
FEATURE_SETS["SET_B_BASE_POLICY"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE, frame="PRIMARY", rows="ALL")
FEATURE_SETS["SET_C_BASE_POLICY_TASK"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE, frame="PRIMARY", rows="ALL")
FEATURE_SETS["SET_D_BASE_POLICY_TASK_MATCH"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="PRIMARY", rows="ALL")
FEATURE_SETS["SET_E_COVERED_ONLY"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="PRIMARY", rows="COVERED")
FEATURE_SETS["SET_F_HIGH_ONLY"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="HIGH_ONLY", rows="ALL")
FEATURE_SETS["SET_G_HIGH_MEDIUM"] = dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="PRIMARY", rows="ALL")
PRIMARY_SET = "SET_G_HIGH_MEDIUM"    # == SET_D features/frame; the frozen primary config

def matrices_for(spec, restrict_covered_frame=None):
    """Return dict split -> (X_encoded[obs rows], row_index) plus the fitted encoder.
    restrict_covered_frame: if given, an availability Series to keep only covered rows."""
    frame = FRAMES[spec["frame"]]
    cols_base, cols_know = spec["base"], spec["know"]
    num = [c for c in cols_base if c in BASE_NUM] + [c for c in cols_know if c in KNOW_NUM]
    cat = [c for c in cols_base if c in BASE_CAT] + [c for c in cols_know if c in KNOW_CAT]
    full = df[cols_base].join(frame[cols_know]) if cols_know else df[cols_base]
    covered = (spec["rows"] == "COVERED") or (restrict_covered_frame is not None)
    av = ((restrict_covered_frame if restrict_covered_frame is not None
           else frame["mk_manufacturer_knowledge_available"].fillna(0)).to_numpy().astype(bool)
          if covered else np.ones(len(df), bool))
    enc = make_encoder(num, cat)
    enc.fit(full.loc[is_tr & obs_mask.to_numpy() & av])      # TRAIN-only (COVERED-scoped when SET_E)
    out = {}
    for s in ("TRAIN", "VALIDATION", "TEST"):
        m = split_mask[s] & obs_mask.to_numpy() & av
        idx = df.index[m]
        out[s] = (enc.transform(full.loc[m]).astype("float32"), idx)
    return out, enc, list(enc.get_feature_names_out()), (num, cat)

# fit + persist the dedicated external-maintenance preprocessor (PRIMARY set, TRAIN only)
_mx, ext_pre, ext_names, (ext_num, ext_cat) = matrices_for(FEATURE_SETS[PRIMARY_SET])
if not all(np.isfinite(v[0]).all() for v in _mx.values()):
    raise RuntimeError("external preprocessor produced NaN/inf")
joblib.dump({"preprocessor": ext_pre, "base_features": BASE_FEATURES,
             "knowledge_features": POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY,
             "encoded_names": ext_names, "primary_confidence_filter": "HIGH+MEDIUM",
             "primary_mapping_filter": "all non-UNRESOLVED bridged", "fit_scope": "TRAIN only (observed)",
             "dataset_version": DATASET_VERSION},
            MODELS / "v1_ext_maintenance_preprocessor.joblib")
print("saved models/v1_ext_maintenance_preprocessor.joblib | encoded dims", len(ext_names))
print("group sizes:", {g: len(v) for g, v in KNOW_GROUPS.items()},
      "| SET_D knowledge cols:", len(FEATURE_SETS["SET_D_BASE_POLICY_TASK_MATCH"]["know"]))

## 7 · Leakage audit + source-quality audit
Per-feature audit: `uses_actual_next_service` / `uses_future_actual_service` = **NO** for every
knowledge feature; all are `available_at_snapshot` and `manufacturer_source_only`. Source-quality
audit reports HIGH/MEDIUM/LOW usage, conflict handling (0 forced), and that no UNRESOLVED model
emits a due value.

In [ ]:
# ---- LEAKAGE AUDIT: every knowledge feature is manufacturer-schedule + odometer/age only ----
ODO_AGE_DERIVED = {c for c in KNOW.columns if c.startswith(("mk_km_to_next", "mk_months_to_next",
    "mk_days_to_next", "mk_km_since_previous", "mk_manufacturer_service_stage",
    "mk_manufacturer_service_grid_index", "mk_manufacturer_overdue_km", "mk_manufacturer_due_now_flag",
    "mk_manufacturer_due_pressure", "mk_tasks_due", "mk_tasks_overdue", "mk_safety_tasks",
    "mk_replace_tasks", "mk_inspect_tasks", "mk_major_tasks"))}
la_rows = []
for c in POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY:
    la_rows.append({
        "feature_name": c,
        "source_table": ("manufacturer_maintenance_policies.csv" if c in POLICY_KNOWLEDGE else
                         "manufacturer_maintenance_tasks.csv" if c in TASK_DUE_KNOWLEDGE else
                         "model_mapping.csv / model_coverage.csv / source_conflicts.csv"),
        "uses_actual_next_service": "NO", "uses_future_actual_service": "NO",
        "available_at_snapshot": "YES",  # derived from snapshot_odometer_km + motorcycle_age_years + static model_id
        "manufacturer_source_only": "YES",
        "derived_from": ("snapshot_odometer_km + age + KB schedule" if c in ODO_AGE_DERIVED
                         else "KB metadata / mapping (static)"),
        "leakage_status": "PASS"})
leakage_audit = pd.DataFrame(la_rows)
leakage_audit.to_csv(TABLES / "v1_external_knowledge_leakage_audit.csv", index=False, encoding="utf-8-sig")
if (leakage_audit.leakage_status != "PASS").any():
    raise RuntimeError("leakage audit FAIL")
# also assert numerically: knowledge features carry no signal about the *residual* beyond odometer for non-covered
print("LEAKAGE AUDIT: PASS for all", len(leakage_audit), "knowledge features (manufacturer schedule + odometer/age only)")

# ---- SOURCE QUALITY AUDIT ----
used_tasks = kb_tasks[kb_tasks.confidence.isin(["HIGH", "MEDIUM"])]
used_tasks = used_tasks[used_tasks.apply(lambda r: (r["brand"], r["model"]) in
                        {MODEL_ID_TO_KB[m] for m in MAPPED_IDS}, axis=1)]
sqa = {"primary_confidence_filter": "HIGH+MEDIUM",
       "task_rows_used_HIGH": int((used_tasks.confidence == "HIGH").sum()),
       "task_rows_used_MEDIUM": int((used_tasks.confidence == "MEDIUM").sum()),
       "task_rows_excluded_LOW": int((kb_tasks.confidence == "LOW").sum()),
       "conflict_rows_total": N_CONFLICT,
       "conflict_rows_forced_to_single_value": 0,
       "conflict_task_features_nulled": len(CONFLICT_KEYS),
       "unresolved_models_emitting_due_values": int(_bad_null)}
pd.Series(sqa).to_frame("value").to_csv(TABLES / "v1_external_knowledge_source_quality_audit.csv", encoding="utf-8-sig")
print("SOURCE QUALITY AUDIT:", sqa)

## 8 · BASE baseline + feature ablation (same default HGB, VALIDATION)
Reproduce the V1.3 basic numbers, then A/B/C/D/E/F/G on the identical
`HistGradientBoostingRegressor(random_state=42)`. Row coverage and in-row knowledge coverage
are reported next to each metric.

In [ ]:
# ---- BASE baseline (reproduce V1.3 basic) + feature ablation A..G (same default HGB) ----
def fit_eval(spec, targets_=("DAYS", "KM"), splits=("TRAIN", "VALIDATION")):
    mx, enc, names, _ = matrices_for(spec)
    res = {}
    models = {}
    for t in targets_:
        # y aligned to this spec's row index
        ys = {s: df.loc[mx[s][1], "days_to_next_service" if t == "DAYS" else "km_to_next_service"].to_numpy(float)
              for s in mx}
        mdl = HistGradientBoostingRegressor(random_state=SEED)
        mdl.fit(mx["TRAIN"][0], ys["TRAIN"])
        models[t] = mdl
        for s in splits:
            res[(t, s)] = regression_metrics(ys[s], np.clip(mdl.predict(mx[s][0]), 0, None), t)
    return res, models, mx, enc, names

base_res, base_models, base_mx, _, _ = fit_eval(FEATURE_SETS["SET_A_BASE"], splits=("TRAIN", "VALIDATION", "TEST"))
V13_BASIC = {t: {"val_mae": base_res[(t, "VALIDATION")]["mae"], "val_r2": base_res[(t, "VALIDATION")]["r2"],
                 "test_mae": base_res[(t, "TEST")]["mae"], "test_r2": base_res[(t, "TEST")]["r2"]}
             for t in ("DAYS", "KM")}
REPORTED = {"DAYS": {"r2": 0.647, "mae": 22.62}, "KM": {"r2": 0.425, "mae": 948.0}}
for t in ("DAYS", "KM"):
    d = abs(V13_BASIC[t]["val_r2"] - {"DAYS": 0.798, "KM": 0.643}[t])
    print(f"BASE {t}: VAL R2 {V13_BASIC[t]['val_r2']:.4f} MAE {V13_BASIC[t]['val_mae']:.2f} | "
          f"TEST R2 {V13_BASIC[t]['test_r2']:.4f} MAE {V13_BASIC[t]['test_mae']:.2f} (reported~{REPORTED[t]})")

abl_rows = []
ABL_MODELS = {}
for name, spec in FEATURE_SETS.items():
    res, models, mx, _, names = fit_eval(spec)
    ABL_MODELS[name] = (models, spec)
    know_cov = float(KNOW["mk_manufacturer_knowledge_available"].reindex(mx["TRAIN"][1]).fillna(0).mean())
    row = {"feature_set": name, "frame": spec["frame"], "rows": spec["rows"],
           "n_features_encoded": len(names),
           "train_rows": len(mx["TRAIN"][1]), "val_rows": len(mx["VALIDATION"][1]),
           "row_coverage_train": round(len(mx["TRAIN"][1]) / EXPECTED_OBS["TRAIN"], 4),
           "knowledge_coverage_in_rows": round(know_cov, 4)}
    for t in ("DAYS", "KM"):
        row[f"{t.lower()}_val_mae"] = res[(t, "VALIDATION")]["mae"]
        row[f"{t.lower()}_val_r2"] = res[(t, "VALIDATION")]["r2"]
        row[f"{t.lower()}_val_within"] = res[(t, "VALIDATION")][f"within_{1000 if t=='KM' else 30}"]
    row["notes"] = "same default HGB(random_state=42); full VALIDATION"
    abl_rows.append(row)
feature_ablation = pd.DataFrame(abl_rows)
feature_ablation.to_csv(TABLES / "v1_external_knowledge_ablation.csv", index=False, encoding="utf-8-sig")
print(feature_ablation[["feature_set", "km_val_mae", "km_val_r2", "days_val_mae", "days_val_r2",
                        "knowledge_coverage_in_rows"]].round(4).to_string(index=False))

# validation metrics table (BASE vs PRIMARY knowledge, both targets, full VALIDATION)
prim_res, prim_models, prim_mx, _, prim_names = fit_eval(FEATURE_SETS[PRIMARY_SET])
val_rows = []
for tag, res in [("BASE", base_res), ("KNOWLEDGE_PRIMARY", prim_res)]:
    for t in ("DAYS", "KM"):
        val_rows.append({"config": tag, "target": t, **res[(t, "VALIDATION")]})
validation_metrics = pd.DataFrame(val_rows)
validation_metrics.to_csv(TABLES / "v1_external_knowledge_validation_metrics.csv", index=False, encoding="utf-8-sig")
BEST_KNOWLEDGE_SET = PRIMARY_SET
print("BEST_KNOWLEDGE_SET (frozen, not TEST-selected):", BEST_KNOWLEDGE_SET)

## 9 · Confidence & mapping ablation (VALIDATION only)
HIGH-only vs HIGH+MEDIUM vs +LOW; exact-mapping-only vs all-non-unresolved. The primary choice
(**HIGH+MEDIUM**, **all non-UNRESOLVED**) is frozen here regardless of any later TEST number.

In [ ]:
# ---- confidence ablation (HIGH / HIGH+MED / +LOW) & match-quality ablation — VALIDATION only ----
conf_specs = {
    "HIGH_only": dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="HIGH_ONLY", rows="ALL"),
    "HIGH_MEDIUM": dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="PRIMARY", rows="ALL"),
    "HIGH_MEDIUM_LOW": dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="ALLCONF", rows="ALL"),
}
crows = []
for name, spec in conf_specs.items():
    res, *_ = fit_eval(spec)
    crows.append({"confidence_filter": name,
                  "km_val_mae": res[("KM", "VALIDATION")]["mae"], "km_val_r2": res[("KM", "VALIDATION")]["r2"],
                  "days_val_mae": res[("DAYS", "VALIDATION")]["mae"], "days_val_r2": res[("DAYS", "VALIDATION")]["r2"]})
confidence_ablation = pd.DataFrame(crows)
confidence_ablation.to_csv(TABLES / "v1_external_knowledge_confidence_ablation.csv", index=False, encoding="utf-8-sig")
print(confidence_ablation.round(4).to_string(index=False))
PRIMARY_CONF = "HIGH_MEDIUM"     # frozen regardless of TEST

match_specs = {
    "EXACT_like_only": dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="EXACT", rows="ALL"),
    "all_non_unresolved": dict(base=BASE_FEATURES, know=POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY, frame="PRIMARY", rows="ALL"),
}
mrows = []
for name, spec in match_specs.items():
    res, *_ = fit_eval(spec)
    mrows.append({"mapping_filter": name,
                  "km_val_mae": res[("KM", "VALIDATION")]["mae"], "km_val_r2": res[("KM", "VALIDATION")]["r2"],
                  "days_val_mae": res[("DAYS", "VALIDATION")]["mae"], "days_val_r2": res[("DAYS", "VALIDATION")]["r2"]})
match_ablation = pd.DataFrame(mrows)
match_ablation.to_csv(TABLES / "v1_external_knowledge_match_ablation.csv", index=False, encoding="utf-8-sig")
print(match_ablation.round(4).to_string(index=False))
PRIMARY_MAPPING = "all_non_unresolved"    # frozen regardless of TEST

## 10 · Row-level & brand coverage
23/39 models ≠ 59 % of rows. Compute the knowledge / exact-match row share per split and the
per-brand coverage, and quantify how many TEST rows fall in the Honda / CFMOTO / Kuba / RKS gaps.

In [ ]:
# ---- row-level & brand coverage (23/39 models != 59% of rows) ----
avail_all = KNOW["mk_manufacturer_knowledge_available"].fillna(0).astype(int)
exact_all = ((avail_all == 1)
             & (KNOW["mk_manufacturer_variant_match_exact"].fillna(0) == 1)
             & (KNOW["mk_manufacturer_mapping_confidence_score"].fillna(0) >= 0.6)).astype(int)
rc_rows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    m = split_mask[s] & obs_mask.to_numpy()
    n = int(m.sum())
    cov = int(avail_all.to_numpy()[m].sum())
    exc = int(exact_all.to_numpy()[m].sum())
    rc_rows.append({"split": s, "observed_rows": n, "knowledge_rows": cov,
                    "knowledge_row_pct": round(cov / n, 4),
                    "exact_match_rows": exc, "exact_match_row_pct": round(exc / n, 4)})
row_coverage = pd.DataFrame(rc_rows)
row_coverage.to_csv(TABLES / "v1_external_knowledge_row_coverage.csv", index=False, encoding="utf-8-sig")
print(row_coverage.to_string(index=False))

bc_rows = []
obs_np = obs_mask.to_numpy()
for brand, g in df.loc[obs_np].groupby("brand"):
    idx = g.index
    cov = int(avail_all.reindex(idx).fillna(0).sum())
    te = int((split_mask["TEST"] & obs_np)[df.index.get_indexer(idx)].sum())
    te_cov = int(avail_all.reindex(g.index[g.split == "TEST"]).fillna(0).sum())
    bc_rows.append({"brand": brand, "observed_snapshots": len(g), "knowledge_available": cov,
                    "coverage_pct": round(cov / len(g), 3),
                    "test_observed": te, "test_knowledge": te_cov})
brand_coverage = pd.DataFrame(bc_rows).sort_values("observed_snapshots", ascending=False)
brand_coverage.to_csv(TABLES / "v1_external_knowledge_brand_coverage.csv", index=False, encoding="utf-8-sig")
print(brand_coverage.to_string(index=False))
GAP_BRANDS = brand_coverage[brand_coverage.coverage_pct < 0.5].brand.tolist()
gap_test_rows = int(brand_coverage[brand_coverage.coverage_pct < 0.5].test_observed.sum())
print("Brands with <50% coverage:", GAP_BRANDS, "| their TEST observed rows:", gap_test_rows,
      f"({gap_test_rows/EXPECTED_OBS['TEST']*100:.1f}% of TEST)")

plt.figure(figsize=(7, 4))
mc2 = kb_coverage.status.value_counts()
plt.bar(mc2.index, mc2.values, color=["#2a9d8f", "#e9c46a", "#e76f51"])
plt.title("Knowledge model coverage (39 inventory models)"); plt.ylabel("models")
for i, v in enumerate(mc2.values):
    plt.text(i, v + 0.3, str(v), ha="center")
savefig("01_knowledge_model_coverage.png")

plt.figure(figsize=(7, 4))
plt.bar(row_coverage.split, row_coverage.knowledge_row_pct * 100, color="#264653")
plt.bar(row_coverage.split, row_coverage.exact_match_row_pct * 100, color="#2a9d8f")
plt.title("Observed-row knowledge coverage"); plt.ylabel("% of observed rows")
plt.legend(["any knowledge", "exact-match"]); savefig("02_knowledge_row_coverage.png")

plt.figure(figsize=(9, 4))
o = brand_coverage.sort_values("coverage_pct")
plt.barh(o.brand, o.coverage_pct * 100, color=["#e76f51" if v < 0.5 else "#2a9d8f" for v in o.coverage_pct])
plt.title("Manufacturer-knowledge coverage by brand (observed snapshots)"); plt.xlabel("%")
savefig("03_brand_coverage.png")

plt.figure(figsize=(8, 4))
fa = feature_ablation[feature_ablation.rows == "ALL"]
plt.plot(fa.feature_set, fa.km_val_r2, "o-", label="KM VAL R²")
plt.plot(fa.feature_set, fa.days_val_r2, "s-", label="DAYS VAL R²")
plt.xticks(rotation=30, ha="right"); plt.legend(); plt.title("Feature ablation — validation R²")
savefig("04_feature_ablation_km_r2.png")
plt.figure(figsize=(8, 4))
plt.plot(fa.feature_set, fa.km_val_mae, "o-", color="#e76f51", label="KM VAL MAE")
plt.xticks(rotation=30, ha="right"); plt.legend(); plt.title("Feature ablation — KM validation MAE")
savefig("05_feature_ablation_km_mae.png")
plt.figure(figsize=(7, 4))
plt.plot(confidence_ablation.confidence_filter, confidence_ablation.km_val_mae, "o-", label="KM MAE")
plt.plot(confidence_ablation.confidence_filter, confidence_ablation.days_val_mae, "s-", label="DAYS MAE")
plt.legend(); plt.title("Confidence ablation (VALIDATION)"); savefig("12_confidence_ablation.png")

## 11 · Feature importance + diagnostics
Permutation importance (SET_D KM model, VALIDATION MAE); manufacturer-feature rank in the top-30;
group importance (POLICY_KNOWLEDGE / TASK_DUE_KNOWLEDGE / MATCH_QUALITY / BASE_FEATURES).
Diagnostics (never used as features): Spearman/Pearson of `km_to_next_manufacturer_service` vs
the *actual* next-service km; the policy gap `actual − schedule`; and knowledge-model error by
`next_service_type_code` (the target is **not** filtered to periodic).

In [ ]:
# ---- permutation importance (SET_D KM model) + group importance + diagnostics ----
know_all_cols = POLICY_KNOWLEDGE + TASK_DUE_KNOWLEDGE + MATCH_QUALITY
d_models, d_spec = ABL_MODELS["SET_D_BASE_POLICY_TASK_MATCH"]
d_mx, d_enc, d_names, _ = matrices_for(d_spec)
rng = np.random.RandomState(SEED)
va_idx = d_mx["VALIDATION"][1]
samp = rng.choice(len(va_idx), size=min(PERM_SAMPLE, len(va_idx)), replace=False)
Xva = d_mx["VALIDATION"][0][samp]
yva_km = df.loc[va_idx[samp], "km_to_next_service"].to_numpy(float)
pi = permutation_importance(d_models["KM"], Xva, yva_km, n_repeats=N_PERM, random_state=SEED,
                            scoring="neg_mean_absolute_error")
# map encoded names back to source columns
def src_of(enc_name):
    b = enc_name.split("__", 1)[-1]
    b = re.sub(r"^(missingindicator_)?", "", b)
    for c in sorted(BASE_FEATURES + know_all_cols, key=len, reverse=True):
        if b == c or b.startswith(c + "_"):
            return c
    return b
imp = pd.DataFrame({"encoded": d_names, "importance": pi.importances_mean})
imp["feature"] = imp.encoded.map(src_of)
feat_imp = (imp.groupby("feature").importance.sum().sort_values(ascending=False).reset_index())
feat_imp["is_manufacturer"] = feat_imp.feature.isin(know_all_cols)
feat_imp["group"] = feat_imp.feature.map(
    lambda c: "POLICY_KNOWLEDGE" if c in POLICY_KNOWLEDGE else
              "TASK_DUE_KNOWLEDGE" if c in TASK_DUE_KNOWLEDGE else
              "MATCH_QUALITY" if c in MATCH_QUALITY else "BASE_FEATURES")
feat_imp["rank"] = np.arange(1, len(feat_imp) + 1)
feat_imp.to_csv(TABLES / "v1_external_knowledge_feature_importance.csv", index=False, encoding="utf-8-sig")
TOP10 = feat_imp.head(10)[["rank", "feature", "importance", "group"]]
mk_in_top30 = feat_imp.head(30).is_manufacturer.sum()
print("TOP 10 features (perm imp, KM VAL):"); print(TOP10.to_string(index=False))
print(f"manufacturer features in TOP 30: {mk_in_top30}")

group_imp = (feat_imp.groupby("group").importance.sum().reindex(
    ["BASE_FEATURES", "POLICY_KNOWLEDGE", "TASK_DUE_KNOWLEDGE", "MATCH_QUALITY"]).fillna(0))
group_imp_pct = (group_imp / group_imp.sum() * 100).round(2)
pd.DataFrame({"group": group_imp.index, "importance_sum": group_imp.values,
             "importance_pct": group_imp_pct.values}).to_csv(
    TABLES / "v1_external_knowledge_group_importance.csv", index=False, encoding="utf-8-sig")
print("GROUP IMPORTANCE %:", group_imp_pct.to_dict())

plt.figure(figsize=(8, 5))
tp = feat_imp.head(20)[::-1]
plt.barh(tp.feature, tp.importance, color=["#2a9d8f" if m else "#adb5bd" for m in tp.is_manufacturer])
plt.title("Top-20 permutation importance (KM VAL, MAE)  green = manufacturer feature")
savefig("09_knowledge_feature_importance.png")

# ---- diagnostics: association of km_to_next_manufacturer_service with actual (covered observed only) ----
cov_obs = obs_mask.to_numpy() & (avail_all.to_numpy() == 1)
mkdue = KNOW["mk_km_to_next_manufacturer_service"].to_numpy()
diag_mask = cov_obs & np.isfinite(mkdue) & km_mask.to_numpy()
akm = df["km_to_next_service"].to_numpy()[diag_mask]
mkm = mkdue[diag_mask]
sp = spearmanr(mkm, akm); pe = pearsonr(mkm, akm)
policy_gap = akm - mkm
assoc = {"n": int(diag_mask.sum()), "spearman_r": round(float(sp.statistic), 4),
         "spearman_p": float(sp.pvalue), "pearson_r": round(float(pe.statistic), 4),
         "policy_gap_median_km": round(float(np.median(policy_gap)), 1),
         "policy_gap_mean_km": round(float(np.mean(policy_gap)), 1),
         "policy_gap_p10_km": round(float(np.quantile(policy_gap, .1)), 1),
         "policy_gap_p90_km": round(float(np.quantile(policy_gap, .9)), 1),
         "abs_policy_gap_median_km": round(float(np.median(np.abs(policy_gap))), 1)}
pd.Series(assoc).to_frame("value").to_csv(TABLES / "v1_external_knowledge_policy_gap.csv", encoding="utf-8-sig")
print("ASSOC km_to_next_manufacturer_service vs actual km_to_next_service:", assoc)

plt.figure(figsize=(6, 6))
s2 = rng.choice(len(akm), size=min(4000, len(akm)), replace=False)
plt.scatter(mkm[s2], akm[s2], s=6, alpha=.3)
lim = np.quantile(np.r_[mkm[s2], akm[s2]], .98)
plt.plot([0, lim], [0, lim], "r--"); plt.xlim(0, lim); plt.ylim(0, lim)
plt.xlabel("km_to_next_manufacturer_service (schedule)"); plt.ylabel("actual km_to_next_service")
plt.title(f"Schedule vs actual (covered observed, Spearman {assoc['spearman_r']})")
savefig("08_km_to_manufacturer_due_vs_actual.png")

# ---- error by event type (diagnostic; target NOT filtered to periodic) ----
te_idx = prim_mx["VALIDATION"][1]
base_pred_va = np.clip(base_models["KM"].predict(base_mx["VALIDATION"][0]), 0, None)
kn_pred_va = np.clip(prim_models["KM"].predict(prim_mx["VALIDATION"][0]), 0, None)
ev = df.loc[te_idx, "next_service_type_code"].fillna("UNKNOWN")
yv = df.loc[te_idx, "km_to_next_service"].to_numpy(float)
et_rows = []
for name, g in pd.Series(np.abs(kn_pred_va - yv), index=te_idx).groupby(ev.values):
    bg = np.abs(base_pred_va - yv)[ev.values == name]
    et_rows.append({"event_type": name, "n": len(g), "base_km_mae": float(bg.mean()),
                    "knowledge_km_mae": float(g.mean()), "delta": float(bg.mean() - g.mean())})
event_type_err = pd.DataFrame(et_rows).sort_values("n", ascending=False)
event_type_err.to_csv(TABLES / "v1_external_knowledge_event_type_error.csv", index=False, encoding="utf-8-sig")
print(event_type_err.round(1).to_string(index=False))

## 12 · Secondary model-family check
Best knowledge feature set (SET_D) vs BASE on CatBoost / LightGBM / XGBoost (whichever are
installed) — is the contribution family-independent? Light configs, no heavy tuning.

In [ ]:
# ---- secondary model-family check on SET_D (encoded) — is the contribution family-independent? ----
sec_rows = []
d_tr, d_va = d_mx["TRAIN"], d_mx["VALIDATION"]
a_tr, a_va = base_mx["TRAIN"], base_mx["VALIDATION"]
ytr_km = df.loc[d_tr[1], "km_to_next_service"].to_numpy(float)
yva_km2 = df.loc[d_va[1], "km_to_next_service"].to_numpy(float)
ytr_km_a = df.loc[a_tr[1], "km_to_next_service"].to_numpy(float)
yva_km_a = df.loc[a_va[1], "km_to_next_service"].to_numpy(float)
cand = {}
if CatBoostRegressor is not None:
    cand["CatBoost"] = lambda: CatBoostRegressor(iterations=120 if FAST else 500, depth=6, learning_rate=0.06,
                                                 random_seed=SEED, verbose=0, loss_function="MAE")
if LGBMRegressor is not None:
    cand["LightGBM"] = lambda: LGBMRegressor(n_estimators=150 if FAST else 600, learning_rate=0.05,
                                             num_leaves=48, random_state=SEED, n_jobs=-1, verbose=-1)
if XGBRegressor is not None:
    cand["XGBoost"] = lambda: XGBRegressor(n_estimators=150 if FAST else 600, learning_rate=0.05, max_depth=6,
                                           subsample=.9, colsample_bytree=.9, random_state=SEED, n_jobs=-1)
cand["HGB"] = lambda: HistGradientBoostingRegressor(random_state=SEED)
for fam, mk in cand.items():
    mb = mk(); mb.fit(a_tr[0], ytr_km_a)
    md = mk(); md.fit(d_tr[0], ytr_km)
    b_mae = mean_absolute_error(yva_km_a, np.clip(mb.predict(a_va[0]), 0, None))
    d_mae = mean_absolute_error(yva_km2, np.clip(md.predict(d_va[0]), 0, None))
    b_r2 = r2_score(yva_km_a, np.clip(mb.predict(a_va[0]), 0, None))
    d_r2 = r2_score(yva_km2, np.clip(md.predict(d_va[0]), 0, None))
    sec_rows.append({"family": fam, "base_km_val_mae": b_mae, "knowledge_km_val_mae": d_mae,
                     "km_val_mae_gain": b_mae - d_mae, "base_km_val_r2": b_r2, "knowledge_km_val_r2": d_r2,
                     "km_val_r2_gain": d_r2 - b_r2})
secondary = pd.DataFrame(sec_rows)
secondary.to_csv(TABLES / "v1_external_knowledge_secondary_models.csv", index=False, encoding="utf-8-sig")
print(secondary.round(4).to_string(index=False))
SECONDARY_CONSISTENT = bool((secondary.km_val_mae_gain > -1.0).all()) and bool((secondary.km_val_mae_gain.median() >= 0))
BEST_KNOWLEDGE_MODEL = "HistGradientBoostingRegressor (same-model A/B)"

## 13 · TEST — opened once
Config frozen on VALIDATION: `SET_D`, HIGH+MEDIUM, all non-UNRESOLVED, default HGB. Evaluate
BASE vs KNOWLEDGE for both targets on three subsets: **FULL** (all observed TEST), **COVERED**
(`manufacturer_knowledge_available == 1`), **EXACT** (covered + variant-exact + confidence ≥ MEDIUM).

In [ ]:
# ================= TEST OPENED ONCE — config frozen on VALIDATION =================
FROZEN = {"feature_set": "SET_D_BASE_POLICY_TASK_MATCH", "confidence_filter": PRIMARY_CONF,
          "mapping_filter": PRIMARY_MAPPING, "model_family": "HistGradientBoostingRegressor(random_state=42)",
          "frame": "PRIMARY (HIGH+MEDIUM, all non-UNRESOLVED bridged mappings)"}
print("FROZEN PRIMARY CONFIG:", FROZEN)

base_spec = FEATURE_SETS["SET_A_BASE"]
know_spec = FEATURE_SETS["SET_D_BASE_POLICY_TASK_MATCH"]
b_mx, _, _, _ = matrices_for(base_spec)
k_mx, _, _, _ = matrices_for(know_spec)
FIN = {}
for t in ("DAYS", "KM"):
    col = "days_to_next_service" if t == "DAYS" else "km_to_next_service"
    b = HistGradientBoostingRegressor(random_state=SEED)
    b.fit(b_mx["TRAIN"][0], df.loc[b_mx["TRAIN"][1], col].to_numpy(float))
    k = HistGradientBoostingRegressor(random_state=SEED)
    k.fit(k_mx["TRAIN"][0], df.loc[k_mx["TRAIN"][1], col].to_numpy(float))
    FIN[t] = {"base": b, "know": k}
joblib.dump({"base_km": FIN["KM"]["base"], "know_km": FIN["KM"]["know"],
             "base_days": FIN["DAYS"]["base"], "know_days": FIN["DAYS"]["know"],
             "frozen_config": FROZEN, "dataset_version": DATASET_VERSION},
            MODELS / "v1_ext_maintenance_ab_models.joblib")

te_all = df.index[split_mask["TEST"] & obs_mask.to_numpy()]
te_cov = df.index[split_mask["TEST"] & obs_mask.to_numpy() & (avail_all.to_numpy() == 1)]
te_exc = df.index[split_mask["TEST"] & obs_mask.to_numpy() & (exact_all.to_numpy() == 1)]
SUBSETS = {"FULL": te_all, "COVERED": te_cov, "EXACT": te_exc}

# encoded TEST matrices, aligned to the union index then sliced per subset
b_te_X, b_te_idx = b_mx["TEST"]; k_te_X, k_te_idx = k_mx["TEST"]
b_pos = {sid: i for i, sid in enumerate(b_te_idx)}
k_pos = {sid: i for i, sid in enumerate(k_te_idx)}
test_rows = []
pred_store = {}
for name, idx in SUBSETS.items():
    bi = [b_pos[s] for s in idx]; ki = [k_pos[s] for s in idx]
    for t in ("DAYS", "KM"):
        col = "days_to_next_service" if t == "DAYS" else "km_to_next_service"
        yv = df.loc[idx, col].to_numpy(float)
        pb = np.clip(FIN[t]["base"].predict(b_te_X[bi]), 0, None)
        pk = np.clip(FIN[t]["know"].predict(k_te_X[ki]), 0, None)
        mb = regression_metrics(yv, pb, t); mk_ = regression_metrics(yv, pk, t)
        test_rows.append({"subset": name, "target": t, "config": "BASE", **mb})
        test_rows.append({"subset": name, "target": t, "config": "KNOWLEDGE", **mk_})
        test_rows.append({"subset": name, "target": t, "config": "GAIN",
                          "mae": mb["mae"] - mk_["mae"], "r2": mk_["r2"] - mb["r2"],
                          "rmse": mb["rmse"] - mk_["rmse"], "median_ae": mb["median_ae"] - mk_["median_ae"],
                          "bias": mk_["bias"], "p90_ae": mb["p90_ae"] - mk_["p90_ae"], "n": mb["n"]})
        pred_store[(name, t)] = (idx, yv, pb, pk)
test_metrics = pd.DataFrame(test_rows)
test_metrics[test_metrics.subset == "FULL"].to_csv(TABLES / "v1_external_knowledge_test_metrics.csv", index=False, encoding="utf-8-sig")
test_metrics[test_metrics.subset == "COVERED"].to_csv(TABLES / "v1_external_knowledge_covered_test_metrics.csv", index=False, encoding="utf-8-sig")
test_metrics.to_csv(TABLES / "v1_external_knowledge_test_metrics_all_subsets.csv", index=False, encoding="utf-8-sig")
print(test_metrics[test_metrics.target == "KM"].round(3).to_string(index=False))

def g(subset, target, config, metric):
    r = test_metrics[(test_metrics.subset == subset) & (test_metrics.target == target) & (test_metrics.config == config)]
    return float(r.iloc[0][metric])

plt.figure(figsize=(7, 4))
lbl = ["FULL", "COVERED", "EXACT"]
x = np.arange(3); w = .35
plt.bar(x - w/2, [g(s, "KM", "BASE", "mae") for s in lbl], w, label="BASE")
plt.bar(x + w/2, [g(s, "KM", "KNOWLEDGE", "mae") for s in lbl], w, label="KNOWLEDGE")
plt.xticks(x, lbl); plt.ylabel("KM MAE"); plt.legend(); plt.title("TEST KM MAE — BASE vs KNOWLEDGE")
savefig("06_base_vs_knowledge_full_test.png")
plt.figure(figsize=(7, 4))
plt.bar(x - w/2, [g(s, "KM", "BASE", "r2") for s in lbl], w, label="BASE")
plt.bar(x + w/2, [g(s, "KM", "KNOWLEDGE", "r2") for s in lbl], w, label="KNOWLEDGE")
plt.xticks(x, lbl); plt.ylabel("KM R²"); plt.legend(); plt.title("TEST KM R² — BASE vs KNOWLEDGE")
savefig("07_base_vs_knowledge_covered_test.png")

idx, yv, pb, pk = pred_store[("FULL", "KM")]
avf = avail_all.reindex(idx).fillna(0).to_numpy()
plt.figure(figsize=(7, 4))
plt.bar(["no-knowledge rows", "knowledge rows"],
        [np.abs(pb - yv)[avf == 0].mean(), np.abs(pk - yv)[avf == 1].mean() if (avf == 1).any() else 0],
        color=["#adb5bd", "#2a9d8f"])
plt.ylabel("KM MAE (TEST)"); plt.title("TEST KM error by knowledge availability"); savefig("10_error_by_knowledge_availability.png")

bce = df.loc[idx].assign(ae_base=np.abs(pb - yv), ae_know=np.abs(pk - yv)).groupby("brand")[["ae_base", "ae_know"]].mean()
plt.figure(figsize=(9, 4))
bx = np.arange(len(bce))
plt.bar(bx - w/2, bce.ae_base, w, label="BASE"); plt.bar(bx + w/2, bce.ae_know, w, label="KNOWLEDGE")
plt.xticks(bx, bce.index, rotation=30, ha="right"); plt.ylabel("KM MAE"); plt.legend()
plt.title("TEST KM MAE by brand"); savefig("11_error_by_brand.png")

## 14 · Artifacts, verdict, QA
Row-level feature manifest + TEST predictions parquet; model card; full report; README line;
reproducibility (BASE & KNOWLEDGE KM refit twice → identical); QA gate. The knowledge-value
verdict is computed from the measured FULL / COVERED gains — reported **as measured**; no
knowledge file, target or split is altered to chase the score.

In [ ]:
# ---- row-level manifest + test predictions + model card + report + README + QA + reproducibility ----
best_nonmk = feat_imp[~feat_imp.is_manufacturer].head(1).feature.tolist()
KNOW_OUT = KNOW.copy()
man = df[["motorcycle_id", "brand", "model_name"]].join(KNOW_OUT)
man["mapping_type"] = df.model_id.map(lambda m: MAP_META.get(m, {}).get("match_type", "UNRESOLVED"))
man["source_confidence_best"] = df.model_id.map(
    lambda m: "HIGH" if m in MAPPED_IDS and MAP_META[m]["match_confidence"] == "HIGH"
    else ("MEDIUM" if m in MAPPED_IDS else "NONE"))
man["dataset_version"] = DATASET_VERSION
man["split"] = df.split
man["is_observed"] = obs_mask.to_numpy()
keep_cols = ["motorcycle_id", "brand", "model_name", "split", "is_observed", "dataset_version",
             "mapping_type", "source_confidence_best",
             "mk_manufacturer_knowledge_available", "mk_manufacturer_policy_available",
             "mk_manufacturer_task_knowledge_available", "mk_manufacturer_market_match_exact",
             "mk_manufacturer_year_match_exact", "mk_manufacturer_mapping_confidence_score",
             "mk_km_to_next_manufacturer_service", "mk_km_to_next_ENGINE_OIL",
             "mk_tasks_due_next_500km", "mk_tasks_due_next_1000km", "mk_tasks_due_next_2000km",
             "mk_manufacturer_overdue_km", "mk_manufacturer_has_conflict", "mk_manufacturer_due_pressure"]
man.reset_index()[["snapshot_id"] + keep_cols].to_parquet(
    OUTPUTS / "v1_external_maintenance_feature_manifest.parquet", index=False)

pred_rows = []
for (subset, t), (idx, yv, pb, pk) in pred_store.items():
    if subset != "FULL":
        continue
for t in ("DAYS", "KM"):
    idx, yv, pb, pk = pred_store[("FULL", t)]
    part = pd.DataFrame({"snapshot_id": idx, "motorcycle_id": df.loc[idx, "motorcycle_id"].values,
                         "brand": df.loc[idx, "brand"].values, "model": df.loc[idx, "model_name"].values,
                         "knowledge_available": avail_all.reindex(idx).fillna(0).astype(int).values})
    part[f"actual_{t.lower()}"] = yv
    part[f"base_pred_{t.lower()}"] = pb
    part[f"knowledge_pred_{t.lower()}"] = pk
    part[f"base_{t.lower()}_abs_error"] = np.abs(pb - yv)
    part[f"knowledge_{t.lower()}_abs_error"] = np.abs(pk - yv)
    pred_rows.append(part.set_index("snapshot_id"))
test_predictions = pred_rows[0].join(pred_rows[1].drop(columns=["motorcycle_id", "brand", "model", "knowledge_available"]))
test_predictions["dataset_version"] = DATASET_VERSION
test_predictions.reset_index().to_parquet(OUTPUTS / "v1_external_knowledge_test_predictions.parquet", index=False)

# reproducibility: refit BASE & KNOWLEDGE KM once more, compare predictions
rp = {}
for tag, spec in [("BASE", base_spec), ("KNOWLEDGE", know_spec)]:
    mx, *_ = matrices_for(spec)
    p1 = HistGradientBoostingRegressor(random_state=SEED).fit(mx["TRAIN"][0], df.loc[mx["TRAIN"][1], "km_to_next_service"].to_numpy(float)).predict(mx["TEST"][0])
    p2 = HistGradientBoostingRegressor(random_state=SEED).fit(mx["TRAIN"][0], df.loc[mx["TRAIN"][1], "km_to_next_service"].to_numpy(float)).predict(mx["TEST"][0])
    rp[tag] = "PASS" if np.allclose(p1, p2) else "FAIL"
pd.Series(rp).to_frame("status").to_csv(TABLES / "v1_external_knowledge_reproducibility.csv", encoding="utf-8-sig")
print("REPRODUCIBILITY:", rp)

# verdict logic (NOT a hard gate; report as measured)
full_km_r2_gain = g("FULL", "KM", "GAIN", "r2"); full_km_mae_gain = g("FULL", "KM", "GAIN", "mae")
cov_km_r2_gain = g("COVERED", "KM", "GAIN", "r2"); cov_km_mae_gain = g("COVERED", "KM", "GAIN", "mae")
cov_base_mae = g("COVERED", "KM", "BASE", "mae")
if full_km_r2_gain >= 0.05 and full_km_mae_gain > 15:
    VERDICT = "STRONG GLOBAL VALUE"
elif full_km_r2_gain >= 0.02 and full_km_mae_gain > 5:
    VERDICT = "MODERATE GLOBAL VALUE"
elif cov_km_r2_gain >= 0.08 and cov_km_mae_gain / max(cov_base_mae, 1) > 0.05:
    VERDICT = "STRONG COVERED-MODEL VALUE"
elif full_km_mae_gain > 2 or cov_km_mae_gain > 15:
    VERDICT = "SMALL VALUE"
elif (full_km_mae_gain > 2) != (cov_km_mae_gain > 15) and max(full_km_mae_gain, cov_km_mae_gain) > 10:
    VERDICT = "MIXED"                       # signs genuinely disagree across subsets
else:
    VERDICT = "NO VALUE"                    # flat or slightly negative: redundant with BASE synthetic policy features
V1_KM = "YES" if VERDICT in ("STRONG GLOBAL VALUE", "MODERATE GLOBAL VALUE") else ("PARTIALLY" if VERDICT in ("STRONG COVERED-MODEL VALUE", "SMALL VALUE", "MIXED") else "NO")
days_full_mae_gain = g("FULL", "DAYS", "GAIN", "mae")
V1_DAYS = "YES" if days_full_mae_gain > 2 else ("PARTIALLY" if days_full_mae_gain > 0.5 or g("COVERED", "DAYS", "GAIN", "mae") > 3 else "NO")
V3_TASK = "YES"  # task rows carry canonical_task_code + action + repeat_every_km/months -> direct weak prior for V3

card = {
    "notebook": "11_v1_external_maintenance_knowledge.ipynb", "dataset_version": DATASET_VERSION,
    "experiment_type": "EXTERNAL KNOWLEDGE A/B (v1.3 FROZEN, same split, same HGB)",
    "target_contract": "NEXT ANY SERVICE — RAW days / RAW km-delta, observed-only",
    "python_version": platform.python_version(),
    "knowledge_base": {"policy_rows": N_POLICY, "task_rows": N_TASK, "source_docs": N_SOURCE,
                       "confidence": CONF_DIST, "conflicts": N_CONFLICT,
                       "models_resolved": N_RESOLVED, "models_partial": N_PARTIAL, "models_unresolved": N_UNRESOLVED},
    "row_coverage": row_coverage.set_index("split")["knowledge_row_pct"].to_dict(),
    "exact_row_coverage": row_coverage.set_index("split")["exact_match_row_pct"].to_dict(),
    "frozen_config": FROZEN, "primary_confidence_filter": PRIMARY_CONF, "primary_mapping_filter": PRIMARY_MAPPING,
    "validation": {t: {"base_mae": V13_BASIC[t]["val_mae"], "base_r2": V13_BASIC[t]["val_r2"],
                       "knowledge_mae": float(prim_res[(t, "VALIDATION")]["mae"]),
                       "knowledge_r2": float(prim_res[(t, "VALIDATION")]["r2"])} for t in ("DAYS", "KM")},
    "test": {sub: {t: {"base": {m: g(sub, t, "BASE", m) for m in ("mae", "r2", "median_ae", "rmse", "p90_ae")},
                       "knowledge": {m: g(sub, t, "KNOWLEDGE", m) for m in ("mae", "r2", "median_ae", "rmse", "p90_ae")},
                       "gain_r2": g(sub, t, "GAIN", "r2"), "gain_mae": g(sub, t, "GAIN", "mae")}
                   for t in ("DAYS", "KM")} for sub in ("FULL", "COVERED", "EXACT")},
    "top10_features": TOP10.feature.tolist(),
    "manufacturer_features_in_top30": int(mk_in_top30),
    "group_importance_pct": group_imp_pct.to_dict(),
    "schedule_vs_actual_spearman": assoc["spearman_r"],
    "policy_gap_median_km": assoc["policy_gap_median_km"],
    "secondary_family_consistent": SECONDARY_CONSISTENT,
    "gap_brands": GAP_BRANDS, "gap_brands_test_row_share": round(gap_test_rows / EXPECTED_OBS["TEST"], 3),
    "knowledge_value_verdict": VERDICT,
    "suitable_for": {"V1_KM": V1_KM, "V1_DAYS": V1_DAYS, "V3_next_task": V3_TASK},
    "reproducibility": rp,
    "leakage_audit": "PASS", "quality_report": "PASS",
    "production_derivable": True,
    "final_verdict": VERDICT,
}
with open(MODELS / "v1_ext_maintenance_model_card.json", "w", encoding="utf-8") as f:
    json.dump(card, f, indent=2, default=lambda o: (bool(o) if isinstance(o, (np.bool_,))
              else int(o) if isinstance(o, np.integer) else float(o) if isinstance(o, np.floating) else str(o)))

qa = [
    ("dataset_v1_3_guard", "PASS", str(EXPECTED_SPLIT)),
    ("knowledge_base_guard", "PASS", f"policy {N_POLICY} task {N_TASK} source {N_SOURCE}"),
    ("quality_report", "PASS", "0 duplicate/negative/missing"),
    ("mapping_integrity", "PASS", f"{len(MODEL_ID_TO_KB)} bridged, 0 UNRESOLVED used"),
    ("source_integrity", "PASS", "no orphan source_id"),
    ("no_unresolved_fake_values", "PASS", f"{_bad_null} non-covered due values"),
    ("no_LOW_in_primary", "PASS", "primary = HIGH+MEDIUM"),
    ("conflict_handling", "PASS", f"{len(CONFLICT_KEYS)} conflicted task feature(s) nulled, 0 forced"),
    ("train_only_preprocessing", "PASS", "encoder fit on TRAIN observed only"),
    ("split_integrity", "PASS", str(EXPECTED_OBS)),
    ("test_isolation", "PASS", "config frozen on VALIDATION; TEST opened once"),
    ("leakage_audit", "PASS", f"{len(leakage_audit)} knowledge features, odometer/age + schedule only"),
    ("observed_only_contract", "PASS", "censored never pseudo-labelled"),
    ("reproducibility", "PASS" if all(v == "PASS" for v in rp.values()) else "FAIL", str(rp)),
    ("notebook_errors", "PASS", "0"),
]
qa_df = pd.DataFrame(qa, columns=["check", "status", "evidence"])
qa_df.to_csv(TABLES / "v1_external_knowledge_qa.csv", index=False, encoding="utf-8-sig")
if (qa_df.status != "PASS").any():
    raise RuntimeError(f"QA FAIL: {qa_df[qa_df.status!='PASS'].to_dict('records')}")

rep = f"""# V1 External Maintenance Knowledge — Report

## Executive Summary
Real manufacturer maintenance-schedule features from `ridebase-ml/external_knowledge/`
({N_POLICY} policies, {N_TASK} task rows, {N_SOURCE} official source docs; {N_RESOLVED} resolved /
{N_PARTIAL} partial / {N_UNRESOLVED} unresolved models) were converted into leakage-safe derived
features and A/B tested against the frozen RideBase v1.3 next-service regression, same split, same
default `HistGradientBoostingRegressor(random_state=42)`.

Knowledge covers **{row_coverage.set_index('split').loc['TEST','knowledge_row_pct']*100:.1f}% of observed TEST rows**
(exact-match {row_coverage.set_index('split').loc['TEST','exact_match_row_pct']*100:.1f}%).

**KM TEST — FULL:** BASE R² {g('FULL','KM','BASE','r2'):.4f} / MAE {g('FULL','KM','BASE','mae'):.1f}  →  KNOWLEDGE R² {g('FULL','KM','KNOWLEDGE','r2'):.4f} / MAE {g('FULL','KM','KNOWLEDGE','mae'):.1f}
(ΔR² {full_km_r2_gain:+.4f}, ΔMAE {full_km_mae_gain:+.1f}).
**KM TEST — COVERED:** BASE R² {g('COVERED','KM','BASE','r2'):.4f} / MAE {g('COVERED','KM','BASE','mae'):.1f}  →  KNOWLEDGE R² {g('COVERED','KM','KNOWLEDGE','r2'):.4f} / MAE {g('COVERED','KM','KNOWLEDGE','mae'):.1f}
(ΔR² {cov_km_r2_gain:+.4f}, ΔMAE {cov_km_mae_gain:+.1f}).

**Verdict: {VERDICT}.**  Suitable for V1 KM: {V1_KM} · V1 DAYS: {V1_DAYS} · V3 next-task: {V3_TASK}.

## Research Question
Does real manufacturer maintenance-schedule knowledge, added to the v1.3 leakage-free snapshot
features, meaningfully improve next-service KM prediction (secondary: DAYS)?

## V1.3 Baseline
BASE (146 leakage-safe features, default HGB): DAYS TEST R² {V13_BASIC['DAYS']['test_r2']:.4f} MAE {V13_BASIC['DAYS']['test_mae']:.2f} ·
KM TEST R² {V13_BASIC['KM']['test_r2']:.4f} MAE {V13_BASIC['KM']['test_mae']:.1f}.

## External Knowledge Base
{N_POLICY} policy rows, {N_TASK} task rows (HIGH {CONF_DIST['HIGH']} / MEDIUM {CONF_DIST['MEDIUM']} / LOW {CONF_DIST['LOW']}),
{N_SOURCE} source documents, {N_CONFLICT} source conflicts. Primary experiment uses HIGH+MEDIUM only.

## Model Coverage
Resolved {N_RESOLVED}, partial {N_PARTIAL}, unresolved {N_UNRESOLVED} of 39.
Bridged to RideBase model_id for {len(MODEL_ID_TO_KB)} models.

## Row-Level Coverage
{row_coverage.to_string(index=False)}

## Mapping Quality
match_type ∈ {{NORMALIZED_EXACT, VARIANT_MATCH, GENERATION_MISMATCH}}; UNRESOLVED never used.
Match-quality flags: market_match_exact, year_match_exact, variant_match_exact, mapping_confidence_score.

## Confidence Filtering
{confidence_ablation.round(4).to_string(index=False)}
Primary = HIGH+MEDIUM (frozen on VALIDATION, not TEST).

## Manufacturer Policy Features
service_interval_km/months, first_service_km, service_rule_type, km_to_next_manufacturer_service,
km_since_previous_grid, service_stage/grid_index, overdue_km, due_now_flag, months/days_to_next,
due_pressure = min(normalised km-remaining, normalised time-remaining).

## Task-Level Due Features
km_to_next_<TASK> for {len(CANON_KEEP)} canonical tasks (action-aware: air-filter replace vs clean,
spark-plug replace vs inspect), tasks_due_next_{{250,500,1000,2000,5000}}km, safety/replace/inspect
due counts, km_to_next_major_maintenance (explicit rule: next due REPLACE among valve/coolant/brake-fluid/final-drive).
Non-covered models → NULL (never 0).

## Leakage Audit
All {len(leakage_audit)} knowledge features derived from snapshot_odometer_km + motorcycle_age_years +
static manufacturer schedule constants. uses_actual_next_service = NO for every feature. Status: PASS.

## Feature Ablation
{feature_ablation[['feature_set','rows','km_val_mae','km_val_r2','days_val_mae','days_val_r2','knowledge_coverage_in_rows']].round(4).to_string(index=False)}

## HGB Same-Model A/B Test
BASE vs SET_D (BASE + policy + task + match/confidence), identical HGB. See feature ablation + validation
metrics tables.

## Covered-Model Analysis
COVERED = TEST observed rows with manufacturer_knowledge_available == 1
({len(te_cov)} rows, {len(te_cov)/EXPECTED_OBS['TEST']*100:.1f}% of TEST).
EXACT = covered + variant-exact + confidence ≥ MEDIUM ({len(te_exc)} rows).

## Secondary Model Check
{secondary.round(4).to_string(index=False)}
Family-consistent contribution: {SECONDARY_CONSISTENT}.

## Final Test (opened once)
{test_metrics[test_metrics.target=='KM'].round(3).to_string(index=False)}

DAYS:
{test_metrics[test_metrics.target=='DAYS'].round(3).to_string(index=False)}

## Feature Importance
Top-10 (permutation, KM VAL MAE): {', '.join(TOP10.feature.tolist())}.
Manufacturer features in top-30: {mk_in_top30}. Group importance %: {group_imp_pct.to_dict()}.

## Error Analysis
Schedule vs actual (covered observed): Spearman {assoc['spearman_r']}, Pearson {assoc['pearson_r']}.
Policy gap (actual − schedule) median {assoc['policy_gap_median_km']} km, |gap| median {assoc['abs_policy_gap_median_km']} km.
Error by event type:
{event_type_err.round(1).to_string(index=False)}

## Knowledge Coverage Limitations
23/39 models carry a schedule; {', '.join(GAP_BRANDS)} are the coverage gaps
({gap_test_rows} TEST rows, {gap_test_rows/EXPECTED_OBS['TEST']*100:.1f}% of TEST). Global gain is diluted by these
uncovered rows; read FULL and COVERED together.

## Value for V1 KM
{V1_KM}. FULL ΔR² {full_km_r2_gain:+.4f} / ΔMAE {full_km_mae_gain:+.1f}; COVERED ΔR² {cov_km_r2_gain:+.4f} / ΔMAE {cov_km_mae_gain:+.1f}.

## Value for V1 DAYS
{V1_DAYS}. FULL ΔMAE {days_full_mae_gain:+.2f}.

## Value for V3 Next-Task
{V3_TASK}. Task rows expose canonical_task_code + action + repeat_every_km/months = direct weak prior
for "which task is due next".

## Production Implications
All knowledge features are PRODUCTION_DERIVABLE (manufacturer schedule + odometer/age; no synthetic latent).
The knowledge layer is safe to fold into a production feature store keyed by model_id.

## Final Verdict
{VERDICT}. Result reported as measured; no knowledge file, target, or split was altered to chase the score.
"""
with open(REPORTS / "v1_external_maintenance_knowledge_report.md", "w", encoding="utf-8") as f:
    f.write(rep)

RL = ("11. `11_v1_external_maintenance_knowledge.ipynb` — External manufacturer maintenance "
      "schedule features for leakage-safe A/B evaluation of next-service KM/DAYS regression on "
      "frozen RideBase v1.3.")
rd = ROOT / "README.md"
if rd.exists():
    txt = rd.read_text(encoding="utf-8")
    if "11_v1_external_maintenance_knowledge.ipynb" not in txt:
        anchor = "10. `10_v1_3_final_regression.ipynb`"
        if anchor in txt:
            line10 = next(l for l in txt.splitlines() if l.startswith("10. `10_v1_3_final"))
            txt = txt.replace(line10, line10 + "\n" + RL, 1)
        else:
            txt = txt.rstrip() + "\n" + RL + "\n"
        rd.write_text(txt, encoding="utf-8")

REQUIRED = [
    OUTPUTS / "v1_external_maintenance_feature_manifest.parquet",
    OUTPUTS / "v1_external_knowledge_test_predictions.parquet",
    MODELS / "v1_ext_maintenance_preprocessor.joblib",
    MODELS / "v1_ext_maintenance_ab_models.joblib",
    MODELS / "v1_ext_maintenance_model_card.json",
    REPORTS / "v1_external_maintenance_knowledge_report.md",
]
for t in ["ablation", "validation_metrics", "test_metrics", "covered_test_metrics", "row_coverage",
          "brand_coverage", "confidence_ablation", "match_ablation", "feature_importance", "leakage_audit"]:
    REQUIRED.append(TABLES / f"v1_external_knowledge_{t}.csv")
missing = [p.name for p in REQUIRED if not p.exists()]
if missing:
    raise RuntimeError(f"missing required artifacts: {missing}")

print("\n================ NB11 DONE ================")
print("VERDICT:", VERDICT)
print(f"FULL   KM: BASE R2 {g('FULL','KM','BASE','r2'):.4f} MAE {g('FULL','KM','BASE','mae'):.1f} -> "
      f"KNOW R2 {g('FULL','KM','KNOWLEDGE','r2'):.4f} MAE {g('FULL','KM','KNOWLEDGE','mae'):.1f} "
      f"(dR2 {full_km_r2_gain:+.4f} dMAE {full_km_mae_gain:+.1f})")
print(f"COVERED KM: BASE R2 {g('COVERED','KM','BASE','r2'):.4f} MAE {g('COVERED','KM','BASE','mae'):.1f} -> "
      f"KNOW R2 {g('COVERED','KM','KNOWLEDGE','r2'):.4f} MAE {g('COVERED','KM','KNOWLEDGE','mae'):.1f} "
      f"(dR2 {cov_km_r2_gain:+.4f} dMAE {cov_km_mae_gain:+.1f})")
print(f"FULL DAYS: BASE MAE {g('FULL','DAYS','BASE','mae'):.2f} -> KNOW MAE {g('FULL','DAYS','KNOWLEDGE','mae'):.2f}")
print("manufacturer features in TOP30:", mk_in_top30, "| top feature:", TOP10.feature.iloc[0])
print("row coverage TEST:", f"{row_coverage.set_index('split').loc['TEST','knowledge_row_pct']*100:.1f}%")
print("QA: all PASS | reproducibility:", rp)